In [1]:
import sys

sys.path.append("..")
import torch
from model import FF_TE
from config import config

train_set = FF_TE.FF_TE("train")
val_set = FF_TE.FF_TE("val")
test_set = FF_TE.FF_TE("test")
trn_loader = torch.utils.data.DataLoader(
    train_set,
    config.batch_size,
    drop_last=False,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)
val_loader = torch.utils.data.DataLoader(
    val_set,
    config.batch_size,
    drop_last=False,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)
tst_loader = torch.utils.data.DataLoader(
    test_set,
    config.batch_size,
    drop_last=False,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)

In [2]:
import numpy as np
from tqdm import tqdm
from model import FF_RBF, utils

repetitions = 300

model = torch.load("./saveTELog/exp_14/output_model.pth").to(
    torch.device(config.device)
)
inputs, labels = next(iter(tst_loader))
inputs, labels = utils.preprocess_inputs(inputs, labels)
print("warm up ...\n")
with torch.no_grad():
    for _ in range(100):
        _ = model.forward_downstream_classification_model(inputs, labels)

torch.cuda.synchronize()

starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(
    enable_timing=True
)
timings = np.zeros((repetitions, 1))
with torch.no_grad():
    for rep in tqdm(range(repetitions)):
        starter.record()
        _ = model.forward_downstream_classification_model(inputs, labels)
        ender.record()
        torch.cuda.synchronize()
        curr_time = starter.elapsed_time(ender)
        timings[rep] = curr_time

avg = timings.sum() / repetitions
print("\navg time={}\n".format(avg))

total = sum([param.nelement() for param in model.parameters()])
print(f"\nNumber of parameter:{total}")

/ExtHDD/Users/Astroyd/SORBF/writing/../model/FF_TE.py:66: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(class_label), num_classes=self.num_classes
/ExtHDD/Users/Astroyd/SORBF/writing/../model/FF_TE.py:66: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(class_label), num_classes=self.num_classes


warm up ...



100%|██████████| 300/300 [00:00<00:00, 4050.18it/s]


avg time=0.23043391938010852


Number of parameter:96018


In [3]:
import torch.nn as nn
import torch.nn.functional as F


class MLPNet(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super(MLPNet, self).__init__()
        self.model = nn.ModuleList(
            [
                nn.Linear(in_features, hidden_features),
                nn.Linear(hidden_features, out_features),
            ]
        )
        self.relu = nn.ReLU()

    def forward(self, input):
        hidden = self.relu(self.model[0](input))
        return F.softmax(self.model[1](hidden), dim=1)

In [6]:
model = MLPNet(9, 500, 4).cuda()

print("warm up ...\n")
with torch.no_grad():
    for _ in range(100):
        _ = model(inputs["original_sample"])

torch.cuda.synchronize()

starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(
    enable_timing=True
)
timings = np.zeros((repetitions, 1))
with torch.no_grad():
    for rep in tqdm(range(repetitions)):
        starter.record()
        _ = model(inputs["original_sample"])
        ender.record()
        torch.cuda.synchronize()
        curr_time = starter.elapsed_time(ender)
        timings[rep] = curr_time

avg = timings.sum() / repetitions
print("\navg time={}\n".format(avg))

total = sum([param.nelement() for param in model.parameters()])
print(f"\nNumber of parameter:{total}")

warm up ...



100%|██████████| 300/300 [00:00<00:00, 17584.71it/s]


avg time=0.043197759818285705


Number of parameter:7004
